# Auditoría inicial del tamaño de los datos MiBici

Cuenta viajes y estaciones para el periodo de análisis propuesto (octubre de 2025 a septiembre de 2026). La lectura de auditoría se limita a las columnas de estación y no carga ni muestra identificadores de usuario, género, año de nacimiento ni registros individuales. La sección de exportación posterior reúne todos los archivos mensuales disponibles en `data/raw`, independientemente del periodo de análisis. Esta es una auditoría inicial de conteos, no una validación integral de calidad.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while not (project_root / "data" / "raw").is_dir() and project_root != project_root.parent:
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
if not raw_data_dir.is_dir():
    raise FileNotFoundError("No se encontró data/raw desde la ubicación actual del notebook.")

months = pd.period_range(start="2025-10", end="2026-09", freq="M")
trip_files = [raw_data_dir / f"datos_abiertos_{month.strftime('%Y_%m')}.csv" for month in months]
missing_files = [path for path in trip_files if not path.is_file()]
if missing_files:
    missing_list = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(f"Faltan archivos mensuales esperados:\n{missing_list}")

station_file = raw_data_dir / "nomenclatura_2026_09.csv"
if not station_file.is_file():
    raise FileNotFoundError(f"No se encontró el archivo de estaciones: {station_file}")

## Conteo de viajes y estaciones observadas en los viajes

Para cada CSV se leen únicamente `Origen_Id` y `Destino_Id`. El número de filas de esas columnas da el conteo de viajes leídos; los IDs únicos se agregan entre los meses para contar estaciones observadas en origen, destino y en ambos.

In [2]:
monthly_rows = []
origin_ids = set()
destination_ids = set()

for path, month in zip(trip_files, months, strict=True):
    trips = pd.read_csv(
        path,
        usecols=["Origen_Id", "Destino_Id"],
        dtype="string",
        encoding_errors="replace",
    )

    origins = trips["Origen_Id"].str.strip()
    destinations = trips["Destino_Id"].str.strip()
    origin_ids.update(origins.dropna().loc[lambda values: values.ne("")].unique())
    destination_ids.update(
        destinations.dropna().loc[lambda values: values.ne("")].unique()
    )
    monthly_rows.append({"mes": str(month), "viajes_leidos": len(trips)})

monthly_counts = pd.DataFrame(monthly_rows)
total_trips = int(monthly_counts["viajes_leidos"].sum())
trip_station_ids = origin_ids | destination_ids

print(f"Meses con archivo leído: {len(monthly_counts)}")
print(f"Viajes leídos en total: {total_trips:,}")
print(f"Estaciones únicas como origen: {len(origin_ids):,}")
print(f"Estaciones únicas como destino: {len(destination_ids):,}")
print(f"Estaciones únicas observadas (origen o destino): {len(trip_station_ids):,}")
monthly_counts

Meses con archivo leído: 12
Viajes leídos en total: 4,497,877
Estaciones únicas como origen: 471
Estaciones únicas como destino: 471
Estaciones únicas observadas (origen o destino): 471


,mes,viajes_leidos
0,2025-10,420064
1,2025-11,380328
2,2025-12,335420
3,2026-01,357180
4,2026-02,333756
5,2026-03,376243
6,2026-04,351851
7,2026-05,315895
8,2026-06,355303
9,2026-07,388257


## Conteo en la nomenclatura publicada

Se cuenta por separado el total de filas y los IDs distintos del archivo de nomenclatura. Este conteo no implica que todas las estaciones listadas estuvieran activas durante cada mes del periodo.

In [3]:
station_table = pd.read_csv(
    station_file,
    usecols=["id"],
    dtype="string",
    encoding_errors="replace",
)
station_ids = station_table["id"].str.strip().dropna()
station_ids = station_ids.loc[station_ids.ne("")]

station_summary = pd.DataFrame(
    [{
        "archivo": station_file.name,
        "filas_leidas": len(station_table),
        "ids_distintos": station_ids.nunique(),
        "ids_en_viajes_no_en_nomenclatura": len(trip_station_ids - set(station_ids)),
    }]
)
station_summary

,archivo,filas_leidas,ids_distintos,ids_en_viajes_no_en_nomenclatura
0,nomenclatura_2026_09.csv,484,484,2


## Unir los archivos mensuales en un Parquet

Primero se leen y concatenan los 142 archivos mensuales disponibles en un único DataFrame para comprobar que las columnas y filas se integran correctamente. Se conservan las ocho columnas originales (`Viaje_Id`, `Usuario_Id`, `Genero`, `Año_de_nacimiento`, `Inicio_del_viaje`, `Fin_del_viaje`, `Origen_Id`, `Destino_Id`); se añade `mes_archivo` para conocer el CSV de origen. Después de validar el DataFrame, una celda separada escribe `data/processed/raw_data.parquet`. La nomenclatura de estaciones se mantiene como archivo independiente.

In [4]:
import re

all_trip_files = sorted(raw_data_dir.glob("datos_abiertos_*.csv"))
if not all_trip_files:
    raise FileNotFoundError(f"No se encontraron CSV mensuales en {raw_data_dir}")

month_by_file = {}
for path in all_trip_files:
    match = re.fullmatch(r"datos_abiertos_(\d{4})_(\d{2})(?:-\d+)?\.csv", path.name)
    if match is None:
        raise ValueError(f"Nombre de archivo mensual no reconocido: {path.name}")
    year, month = (int(value) for value in match.groups())
    month_period = pd.Period(year=year, month=month, freq="M")
    if month_period in month_by_file.values():
        raise ValueError(f"Hay más de un CSV mensual para {month_period}")
    month_by_file[path] = month_period

files_by_month = {month: path for path, month in month_by_file.items()}
expected_all_months = pd.period_range(
    start=min(files_by_month), end=max(files_by_month), freq="M"
)
missing_all_months = [month for month in expected_all_months if month not in files_by_month]
if missing_all_months:
    missing_labels = ", ".join(str(month) for month in missing_all_months)
    raise FileNotFoundError(f"Faltan CSV mensuales dentro del rango: {missing_labels}")

trip_columns = [
    "Viaje_Id",
    "Usuario_Id",
    "Genero",
    "Año_de_nacimiento",
    "Inicio_del_viaje",
    "Fin_del_viaje",
    "Origen_Id",
    "Destino_Id",
]
monthly_frames = []
monthly_dataframe_rows = []

for month_period in expected_all_months:
    path = files_by_month[month_period]
    with path.open("rb") as source_file:
        header_bytes = source_file.readline()
    try:
        header_bytes.decode("utf-8-sig")
        source_encoding = "utf-8-sig"
    except UnicodeDecodeError:
        source_encoding = "cp1252"

    source_header = pd.read_csv(path, nrows=0, encoding=source_encoding)
    source_header = source_header.rename(
        columns={"A}äe_nacimiento": "Año_de_nacimiento"}
    )
    if set(source_header.columns) != set(trip_columns):
        raise ValueError(
            f"Columnas inesperadas en {path.name}: {list(source_header.columns)}"
        )

    month_dataframe = pd.read_csv(
        path,
        dtype="string[pyarrow]",
        encoding=source_encoding,
    ).rename(columns={"A}äe_nacimiento": "Año_de_nacimiento"})
    month_dataframe = month_dataframe.reindex(columns=trip_columns)
    month_dataframe["mes_archivo"] = str(month_period)
    monthly_frames.append(month_dataframe)
    monthly_dataframe_rows.append(
        {"mes": str(month_period), "viajes": len(month_dataframe)}
    )

raw_data_df = pd.concat(monthly_frames, ignore_index=True, copy=False)
expected_columns = trip_columns + ["mes_archivo"]
if list(raw_data_df.columns) != expected_columns:
    raise ValueError(f"El DataFrame unido tiene columnas inesperadas: {list(raw_data_df.columns)}")
if raw_data_df["mes_archivo"].nunique() != len(expected_all_months):
    raise ValueError("El DataFrame no contiene exactamente un valor por cada mes esperado")
if len(raw_data_df) != sum(row["viajes"] for row in monthly_dataframe_rows):
    raise ValueError("El total de filas del DataFrame no coincide con la suma mensual")

monthly_dataframe_counts = pd.DataFrame(monthly_dataframe_rows)
print(f"Archivos concatenados: {len(all_trip_files)}")
print(f"Meses cubiertos: {expected_all_months[0]} a {expected_all_months[-1]}")
print(f"Filas del DataFrame: {len(raw_data_df):,}")
print(f"Columnas del DataFrame: {', '.join(raw_data_df.columns)}")
print(f"Memoria del DataFrame: {raw_data_df.memory_usage(deep=True).sum() / (1024 ** 3):,.2f} GiB")
monthly_dataframe_counts

Archivos concatenados: 142
Meses cubiertos: 2014-12 a 2026-09
Filas del DataFrame: 39,078,984
Columnas del DataFrame: Viaje_Id, Usuario_Id, Genero, Año_de_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id, mes_archivo


Memoria del DataFrame: 6.65 GiB


,mes,viajes
0,2014-12,23967
1,2015-01,32144
2,2015-02,32787
3,2015-03,36096
4,2015-04,34863
...,...,...
137,2026-05,315895
138,2026-06,355303
139,2026-07,388257
140,2026-08,393524


El Parquet se escribe solo después de que el DataFrame anterior pasa sus verificaciones. Se vuelve a abrir el archivo creado y se contrasta el número de filas y el esquema antes de reemplazar el resultado final.

In [5]:
import pyarrow.parquet as pq

output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
parquet_path = output_dir / "raw_data.parquet"
temporary_parquet_path = parquet_path.with_name(f".{parquet_path.name}.tmp")
expected_dataframe_rows = len(raw_data_df)
expected_dataframe_schema = list(raw_data_df.columns)

try:
    raw_data_df.to_parquet(
        temporary_parquet_path,
        engine="pyarrow",
        compression="snappy",
        index=False,
    )
    parquet_metadata = pq.ParquetFile(temporary_parquet_path).metadata
    parquet_schema = pq.ParquetFile(temporary_parquet_path).schema_arrow
    if parquet_metadata.num_rows != expected_dataframe_rows:
        raise ValueError(
            f"El Parquet tiene {parquet_metadata.num_rows:,} filas; "
            f"el DataFrame tiene {expected_dataframe_rows:,}"
        )
    if parquet_schema.names != expected_dataframe_schema:
        raise ValueError(
            f"El esquema Parquet no coincide con el DataFrame: {parquet_schema.names}"
        )
    temporary_parquet_path.replace(parquet_path)
finally:
    temporary_parquet_path.unlink(missing_ok=True)

print(f"Parquet validado: {parquet_path.relative_to(project_root)}")
print(f"Filas: {parquet_metadata.num_rows:,}")
print(f"Columnas: {', '.join(parquet_schema.names)}")
print(f"Tamaño: {parquet_path.stat().st_size / (1024 ** 2):,.1f} MiB")

Parquet validado: data/processed/raw_data.parquet
Filas: 39,078,984
Columnas: Viaje_Id, Usuario_Id, Genero, Año_de_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id, mes_archivo
Tamaño: 737.2 MiB


## Vista previa del Parquet

Lee únicamente las primeras filas para confirmar visualmente la estructura; no carga el archivo completo ni muestra más que una pequeña muestra.

In [5]:
import pandas as pd
from pathlib import Path

In [7]:
parquet_path = "/Users/beibibumer/Documents/GitHub/TC5061-Proyecto-Equipo25/data/processed/raw_data.parquet"

parquet_preview = pd.read_parquet(parquet_path).head()
parquet_preview

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,mes_archivo
0,4601,10324,M,<NA>,2014-12-01 00:33:47,2014-12-01 00:36:54,47,47,2014-12
1,4604,10324,M,<NA>,2014-12-01 01:06:54,2014-12-01 01:08:45,5,5,2014-12
2,4628,3,F,1990.0,2014-12-01 09:47:20,2014-12-01 09:47:47,79,79,2014-12
3,4631,102,M,1982.0,2014-12-01 09:48:23,2014-12-01 09:48:37,79,79,2014-12
4,4632,3,F,1990.0,2014-12-01 09:48:46,2014-12-01 09:48:57,79,79,2014-12


## Interpretación y limitaciones

- Los conteos se calculan sobre las filas que pandas puede leer en los archivos presentes; no se han auditado duplicados, fechas inválidas ni otros errores de calidad.
- El conjunto de IDs observados en los viajes y el de la nomenclatura pueden diferir. La nomenclatura de septiembre de 2026 es una fotografía actual y su correspondencia con las estaciones históricas debe validarse antes de usarla como descripción del periodo completo.
- Las salidas son agregados. No se imprimen datos de viajes ni identificadores personales.